In [63]:
# Hyperparameter tuning

# === Setup ===
!pip install lightgbm pandas scikit-learn matplotlib

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, FunctionTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from lightgbm import LGBMClassifier

# === Parameters ===
CSV_FILE = "virus_data.csv"  # Should include: id, type, family, length, sequence
KMER_SIZE = 6
RANDOM_STATE = 42
N_SPLITS = 5

# === K-mer Extraction ===
def get_kmers(sequence, k):
    sequence = sequence.upper()
    return ' '.join([sequence[i:i+k] for i in range(len(sequence) - k + 1)]) if len(sequence) >= k else ''

# === Load Data from CSV ===
def load_data(csv_path, k):
    df = pd.read_csv(csv_path)
    df['sequence'] = df['sequence'].str.upper().str.replace(r'[^ACGT]', '', regex=True)
    df['kmers'] = df['sequence'].apply(lambda seq: get_kmers(seq, k))
    df = df[df['kmers'].str.len() > 0].copy()

    le = LabelEncoder()
    y = le.fit_transform(df['type'])

    return df['kmers'], y, df['id'], le

# === LightGBM Pipeline + Hyperparameter Tuning ===
def run_lightgbm_pipeline(kmers_train, y_train):
    pipeline = Pipeline([
        ("vectorizer", CountVectorizer()),
        ("to_dense", FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
        ("classifier", LGBMClassifier(random_state=RANDOM_STATE))
    ])

    param_grid = {
        "classifier__n_estimators": [100]
        "classifier__max_depth": [5]
        "classifier__learning_rate": [0.01]
        "classifier__num_leaves": [31]
        "classifier__subsample": [0.6]
        "classifier__colsample_bytree": [0.6]
        "classifier__reg_alpha": [0.0]
        "classifier__reg_lambda": [0.0]
    }

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=2
    )
    grid.fit(kmers_train, y_train)

    print("\n=== Best LightGBM Model ===")
    print("Best F1 Score:", grid.best_score_)
    print("Best Parameters:", grid.best_params_)

    return grid.best_estimator_

# === Evaluation ===
def evaluate_model(model, kmers_test, y_test, label_encoder):
    y_pred = model.predict(kmers_test)
    print("\n=== Test Evaluation ===")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_).plot(cmap="Blues")
    plt.title("Confusion Matrix")
    plt.show()

# === Main ===
def main():
    kmers, y, ids, label_encoder = load_data(CSV_FILE, k=KMER_SIZE)

    kmers_train, kmers_test, y_train, y_test = train_test_split(
        kmers, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    best_model = run_lightgbm_pipeline(kmers_train, y_train)
    evaluate_model(best_model, kmers_test, y_test, label_encoder)

if __name__ == "__main__":
    main()


Fitting 5 folds for each of 96 candidates, totalling 480 fits


KeyboardInterrupt: 

In [63]:
# Hyperparameter tuning

# === Setup ===
!pip install lightgbm pandas scikit-learn matplotlib

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, FunctionTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from lightgbm import LGBMClassifier

# === Parameters ===
CSV_FILE = "virus_data.csv"  # Should include: id, type, family, length, sequence
KMER_SIZE = 6
RANDOM_STATE = 42
N_SPLITS = 5

# === K-mer Extraction ===
def get_kmers(sequence, k):
    sequence = sequence.upper()
    return ' '.join([sequence[i:i+k] for i in range(len(sequence) - k + 1)]) if len(sequence) >= k else ''

# === Load Data from CSV ===
def load_data(csv_path, k):
    df = pd.read_csv(csv_path)
    df['sequence'] = df['sequence'].str.upper().str.replace(r'[^ACGT]', '', regex=True)
    df['kmers'] = df['sequence'].apply(lambda seq: get_kmers(seq, k))
    df = df[df['kmers'].str.len() > 0].copy()

    le = LabelEncoder()
    y = le.fit_transform(df['type'])

    return df['kmers'], y, df['id'], le

# === LightGBM Pipeline + Hyperparameter Tuning ===
def run_lightgbm_pipeline(kmers_train, y_train):
    pipeline = Pipeline([
        ("vectorizer", CountVectorizer()),
        ("to_dense", FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)),
        ("classifier", LGBMClassifier(random_state=RANDOM_STATE))
    ])

    param_grid = {
        "classifier__n_estimators": [100, 200, 500],
        "classifier__max_depth": [5, 10, 20, -1],
        "classifier__learning_rate": [0.01, 0.05, 0.1],
        "classifier__num_leaves": [31, 64, 128],
        "classifier__subsample": [0.6, 0.8, 1.0],
        "classifier__colsample_bytree": [0.6, 0.8, 1.0],
        "classifier__reg_alpha": [0.0, 0.1, 0.5],
        "classifier__reg_lambda": [0.0, 1.0, 5.0]
    }

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=2
    )
    grid.fit(kmers_train, y_train)

    print("\n=== Best LightGBM Model ===")
    print("Best F1 Score:", grid.best_score_)
    print("Best Parameters:", grid.best_params_)

    return grid.best_estimator_

# === Evaluation ===
def evaluate_model(model, kmers_test, y_test, label_encoder):
    y_pred = model.predict(kmers_test)
    print("\n=== Test Evaluation ===")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_).plot(cmap="Blues")
    plt.title("Confusion Matrix")
    plt.show()

# === Main ===
def main():
    kmers, y, ids, label_encoder = load_data(CSV_FILE, k=KMER_SIZE)

    kmers_train, kmers_test, y_train, y_test = train_test_split(
        kmers, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    best_model = run_lightgbm_pipeline(kmers_train, y_train)
    evaluate_model(best_model, kmers_test, y_test, label_encoder)

if __name__ == "__main__":
    main()


Fitting 5 folds for each of 96 candidates, totalling 480 fits


KeyboardInterrupt: 